# Organic Reaction-Mechanism Specialist — Train & Benchmark (Colab GPU)

Fine-tune a small **Qwen** model on the *decontaminated* mechanism dataset built by the
`rxndata` pipeline, then score it on the held-out **oMe-Gold** test set with the **exact
oMeS metric** the oMeBench paper uses — so the number is directly comparable to frontier
models (Gemini, GPT-5.x, Claude, …).

This notebook runs the repo's **full SOTA recipe** (see `docs/sota_features.md`):

**SFT → RL**, with every evidence-linked lever wired in:
1. Build the training set **offline** from the committed Tier-A files (oMe-Silver + oMe-Template),
   run validate → **decontaminate** → dedup, then `training.build_sft_data` with
   **SMILES augmentation** (randomized inputs, canonical targets). Rows carry `messages`
   **+ `prompt` + `reference`** so the same file drives both SFT and RL.
2. *(optional, off by default)* **Distill long CoT** from a frontier model, keeping only
   traces that pass the oMeS verifier — the single biggest quality lever.
3. **SFT** a Qwen base with **DoRA** (weight-decomposed LoRA) and an easy→hard **curriculum**.
4. **RL** — **GRPO or DAPO** against the verifiable **oMeS reward** (multiplicative format
   gate + explicit validity term), with a **validity-collapse monitor**. This is the step
   the oMeBench paper left untapped and the main lever for a small specialist.
5. Evaluate SFT *and* RL checkpoints on **oMe-Gold (196 rxns, never seen in training)** with
   the real oMeS scorer, and drop the scores into a **leaderboard vs frontier models**.

> **Runtime → change type to GPU.** `Runtime ▸ Change runtime type ▸ T4 GPU` (free) or A100/L4 (Pro).
> RL (cell 10) is heavier than SFT; on a free T4 keep `QUICK_MODE=True`.

### ⏱️ Per-cell time estimates

| # | Cell | T4 (free) | A100 / L4 (Pro) |
|---|------|-----------|-----------------|
| 1 | GPU / runtime check | <5 s | <5 s |
| 2 | Install dependencies (+ trl) | 4–7 min | 4–7 min |
| 3 | Get the repo | 10–30 s | 10–30 s |
| 4 | Build decontaminated + augmented dataset | 3–6 min | 3–6 min |
| 5 | Inspect dataset + token lengths | 20–40 s | 20–40 s |
| 6 | *(optional)* Distill frontier CoT | off by default | off by default |
| 7 | Config knobs | <5 s | <5 s |
| 8 | Load base model + tokenizer | 1–3 min (download) | 1–3 min |
| 9 | **Train (DoRA SFT)** | **QUICK ~20 min / full ~60–110 min** | **QUICK ~6 min / full ~12–25 min** |
| 10 | **RL (GRPO/DAPO vs oMeS)** | **QUICK ~25–45 min** | **QUICK ~8–15 min / full ~30–60 min** |
| 11 | Evaluate fine-tuned (SFT+RL) on oMe-Gold | QUICK ~8 min / full ~25–40 min | QUICK ~3 min / full ~6–12 min |
| 12 | (optional) Evaluate the base model | same as #11 | same as #11 |
| 13 | Leaderboard vs frontier | <5 s | <5 s |
| 14 | (optional) Live frontier eval via API | ~5–15 min | ~5–15 min |

`QUICK_MODE = True` (cell 7) subsets training/eval for a fast end-to-end pass; set it `False`
for the full, reportable run. `RUN_RL = True` runs the RL stage; set it `False` to stop after SFT.


## 1 · GPU / runtime check  ·  ⏱️ <5 s
If this prints `No GPU`, enable one via *Runtime ▸ Change runtime type*.

In [ ]:
import subprocess, torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap  = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | compute {cap[0]}.{cap[1]} | VRAM {vram:.0f} GB")
    # bf16 needs Ampere+ (compute >= 8.0); T4 is 7.5 -> use fp16.
    BF16 = cap[0] >= 8
    print("Using", "bf16" if BF16 else "fp16")
else:
    BF16 = False
    print("No GPU — set Runtime ▸ Change runtime type ▸ GPU before continuing.")

## 2 · Install dependencies  ·  ⏱️ 4–7 min
Colab already ships `torch`. We add RDKit (validity/oMeS), transformers/peft/accelerate + **trl**
(SFT + GRPO/DAPO RL), and datasets. Pinned for reproducibility — `trl==0.22.2` is the newest
release compatible with `transformers==4.55.2` that still exposes the DAPO knobs.

In [ ]:
%pip -q install \
  "rdkit==2025.9.2" \
  "transformers==4.55.2" \
  "trl==0.22.2" \
  "peft==0.13.2" \
  "accelerate==1.10.1" \
  "datasets==3.6.0" \
  "pyyaml==6.0.2" "polars==1.36.1" "pyarrow==17.0.0" "tqdm==4.67.1"
print("deps installed — if Colab asks to RESTART the runtime, do it, then re-run from cell 3.")
import transformers, accelerate, peft, trl
print("transformers", transformers.__version__, "| accelerate", accelerate.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)

# trl 0.22.2 gives us DAPO: GRPOConfig(loss_type="dapo", epsilon_high, mask_truncated_completions)
# AND it requires transformers>=4.55.0, so it is compatible with the 4.55.2 pin above.
# peft 0.13+ is needed for DoRA (use_dora=True).

# Guard: transformers 4.55 needs accelerate >= 1.3 (keep_torch_compile kwarg).
# If the *live* accelerate is older than what we just pinned, the runtime is still
# holding Colab's pre-installed version — RESTART before training (Runtime ▸ Restart).
from packaging.version import parse as _v
if _v(accelerate.__version__) < _v("1.3.0"):
    raise RuntimeError(
        f"Live accelerate is {accelerate.__version__} (<1.3). Colab is still using its "
        "pre-installed copy. Do: Runtime ▸ Restart session, then re-run from cell 3.")

## 3 · Get the repo  ·  ⏱️ 10–30 s
Point `REPO_URL` at your clone of this project (it contains the pipeline + the oMeBench data
+ the oMeS scorer). If the repo is private, either make a public mirror, use a token URL, or
upload it and set `REPO_DIR` to the uploaded path.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/YOURNAME/SLM-1.git"   # <-- EDIT ME
REPO_DIR = "/content/SLM-1"

if not os.path.isdir(REPO_DIR):
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    except Exception as e:
        print("git clone failed:", e)
        print("Alternative: upload the repo folder to /content/SLM-1, or mount Drive:")
        print("  from google.colab import drive; drive.mount('/content/drive')")
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
sys.path.insert(0, REPO_DIR)
os.environ["PYTHONPATH"] = os.path.join(REPO_DIR, "src")
print("repo:", REPO_DIR)
print("data files present:", sorted(p.name for p in pathlib.Path("data").glob("oMe_*")))

## 4 · Build the decontaminated + augmented training set  ·  ⏱️ 3–6 min
Runs the pipeline **offline** using only the committed Tier-A files (oMe-Silver + oMe-Template)
to *prove* zero oMe-Gold overlap, then builds the SFT/RL rows with `training.build_sft_data`:

`ingest → normalize → decompose/type → validate → decontaminate(vs oMe-Gold) → dedup`
then `build_sft_data --style cot --augment N` →
`training/data/sft_silver_cot_{train,val}.jsonl`

Two things make this file drive **both** stages:
- every row carries `messages` (SFT target) **and** `prompt` + `reference`
  (`[subtype, canonical_smiles, weight]`) for the oMeS **RL reward**;
- `--augment N` adds N randomized-SMILES **input** variants per train reaction (canonical
  targets) — Bjerrum enumeration — and **decontamination runs after augmentation**, so every
  emitted variant is individually screened against oMe-Gold.

In [ ]:
import subprocess, os, sys, json
env = dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src") + os.pathsep + REPO_DIR)

def run_mod(mod, *args, keep=("GATE","records","PASS","FAIL","removed","kept","leaks",
                              "train /","self-check","tokenizer","Tier","[build]",
                              "[augment]","[decontaminate]")):
    print(f"\n=== {mod} {' '.join(args)} ===")
    r = subprocess.run([sys.executable, "-m", mod, *args],
                       env=env, capture_output=True, text=True)
    for line in (r.stdout + r.stderr).splitlines():
        if any(k in line for k in keep):
            print(line)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise RuntimeError(f"{mod} failed")

# --- Pipeline: proves ZERO oMe-Gold leakage (phase 6) ---
run_mod("rxndata.ontology")
run_mod("rxndata.phase1_ingest", "--only", "ome_silver", "ome_template")
run_mod("rxndata.phase2_normalize", "--only", "ome_silver", "ome_template")
run_mod("rxndata.phase3_mechanism")
run_mod("rxndata.phase5_validate")
run_mod("rxndata.phase6_decontaminate")     # <-- proves 0 gold leaks
run_mod("rxndata.phase7_dedup")

# --- Build SFT/RL rows: messages + prompt + reference, SMILES-augmented, decontaminated ---
# AUGMENT = randomized-SMILES input variants per TRAIN reaction (0 = off).
AUGMENT = 3
run_mod("training.build_sft_data",
        "--dataset", "silver", "--style", "cot",
        "--out-dir", "training/data", "--val-frac", "0.03",
        "--augment", str(AUGMENT))   # decontamination runs AFTER augmentation (default on)

TRAIN_JSONL = "training/data/sft_silver_cot_train.jsonl"
VAL_JSONL   = "training/data/sft_silver_cot_val.jsonl"
print("\nDONE — SFT/RL data in training/data/  (messages + prompt + reference per row)")

## 5 · Inspect dataset + token lengths  ·  ⏱️ 20–40 s
Load the `train`/`val` JSONLs built above, confirm the **decontamination proof**, and
sanity-check token lengths against the Qwen tokenizer (so `MAX_SEQ_LEN` doesn't truncate
long mechanism CoT).

In [ ]:
import json, numpy as np
from collections import Counter

train_rows = [json.loads(l) for l in open(TRAIN_JSONL)]
val_rows   = [json.loads(l) for l in open(VAL_JSONL)]
print(f"train: {len(train_rows)} | val: {len(val_rows)}")
print("levels (train):", dict(Counter(r.get("level") for r in train_rows)))

# Every row carries the RL reward reference — verify before we rely on it in cell 10.
assert all("reference" in r and "prompt" in r for r in train_rows), "missing prompt/reference!"
print("rows carry {messages, prompt, reference, level}: OK  (drives SFT + RL)")

# Decontamination proof (from phase 6, which ran in cell 4).
decon = json.load(open("data/final/decontamination_report.json"))
print("\nDECONTAMINATION vs oMe-Gold:",
      "removed", decon["removed"],
      "| gold InChIKey leaks:", decon.get("post_check_gold_inchikey_leaks"))

# Token-length histogram vs the Qwen tokenizer -> pick MAX_SEQ_LEN so we don't
# silently truncate long mechanism CoT (the reason SFT defaults to 6000).
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", trust_remote_code=True)
lens = [len(_tok.apply_chat_template(r["messages"], tokenize=True)) for r in train_rows[:1500]]
lens = np.array(lens)
print(f"\ntoken lengths (full chat): p50={int(np.percentile(lens,50))} "
      f"p95={int(np.percentile(lens,95))} p99={int(np.percentile(lens,99))} max={lens.max()}")
print("-> set MAX_SEQ_LEN (cell 7) at/above p99 to avoid truncating mechanisms.")

# One rendered example
ex = train_rows[0]
print("\n--- example (user, truncated) ---\n", ex["messages"][1]["content"][:280])
print("\n--- assistant target (tail) ---\n", ex["messages"][2]["content"][-200:])
print("\n--- reward reference (first 2 steps) ---\n", ex["reference"][:2])

## 6 · *(optional)* Distill long CoT from a frontier model  ·  ⏱️ off by default

**The single biggest quality lever** (ether0, RetroDFM-R): warm-start on long reasoning traces
distilled from a frontier model — but keep a trace **only if its final answer passes the oMeS
verifier** (`S_partial ≥ keep-threshold` and full validity). `training/distill_cot.py` does the
answer-conditioned prompting, scoring, rejection filter, and decontamination, and writes rows in
the *same schema* as cell 4.

Off by default (needs an API key + spend). Set `RUN_DISTILL=True` and add a key. When on, the
distilled rows are **merged into the SFT train set** so cell 9 trains on silver + verified CoT.

In [ ]:
RUN_DISTILL   = False          # set True to distill (needs an API key + spend)
DISTILL_MODEL = "gpt-5.5"      # provider alias in omebench_eval.providers (opus, gpt-5.5, ...)
DISTILL_LIMIT = 300            # source reactions to attempt
DISTILL_KEEP  = 0.9            # min oMeS S_partial to keep a trace

if RUN_DISTILL:
    import os
    os.environ.setdefault("OPENAI_API_KEY", "")      # <-- your key (or ANTHROPIC_API_KEY)
    # os.environ.setdefault("ANTHROPIC_API_KEY", "")
    %pip -q install openai anthropic
    run_mod("training.distill_cot",
            "--dataset", "silver", "--model", DISTILL_MODEL,
            "--limit", str(DISTILL_LIMIT), "--keep-threshold", str(DISTILL_KEEP),
            "--out-dir", "training/data")

    # Merge verified distilled traces into the SFT train set (val stays as-is).
    import json
    distilled = "training/data/sft_distilled_cot_train.jsonl"
    extra = [json.loads(l) for l in open(distilled)]
    with open(TRAIN_JSONL, "a") as f:
        for r in extra:
            f.write(json.dumps(r) + "\n")
    train_rows = [json.loads(l) for l in open(TRAIN_JSONL)]
    print(f"merged {len(extra)} distilled traces -> train now {len(train_rows)} rows")
else:
    print("distillation skipped (RUN_DISTILL=False). SFT will use silver + augmented data only.")

## 7 · Config  ·  ⏱️ <5 s
`QUICK_MODE=True` gives a fast full pass (subset + 1 epoch). Set `False` for the reportable run.
`RUN_RL=True` runs the GRPO/DAPO stage after SFT. On a **T4**, if you hit OOM use
`Qwen/Qwen2.5-0.5B-Instruct`, lower `MAX_SEQ_LEN`, or reduce `RL_NUM_GEN`.

In [ ]:
QUICK_MODE      = True                         # False = full, reportable run

BASE_MODEL      = "Qwen/Qwen2.5-1.5B-Instruct" # T4-safe with LoRA/DoRA; use 0.5B if OOM
MAX_SEQ_LEN     = 3072                          # >= p99 from cell 5 so CoT isn't truncated
EPOCHS          = 1 if QUICK_MODE else 3
PER_DEVICE_BATCH= 2
GRAD_ACCUM      = 8                             # effective batch = 16
LR              = 2e-4                           # LoRA/DoRA lr
LORA_R          = 16
LORA_ALPHA      = 32
USE_DORA        = True                           # DoRA (weight-decomposed LoRA); +reasoning vs LoRA
CURRICULUM      = True                           # order SFT easy -> medium -> hard
MAX_TRAIN       = 600 if QUICK_MODE else None    # cap training rows in quick mode

# --- RL (GRPO/DAPO vs the verifiable oMeS reward) ---
RUN_RL          = True                           # False = stop after SFT
RL_ALGO         = "dapo"                          # "dapo" (token-level, clip-higher) or "grpo"
RL_REWARD       = "omes"                           # "omes" or "omes+roundtrip"
RL_NUM_GEN      = 4                                # rollouts per prompt (GRPO group size)
RL_LR           = 1e-6
RL_MAX_PROMPT   = 1024
RL_MAX_COMPL    = 1024
RL_STEPS        = 40 if QUICK_MODE else 300        # optimizer steps (max_steps)
RL_TRAIN_ROWS   = 200 if QUICK_MODE else None      # cap RL prompts in quick mode
VALIDITY_FLOOR  = 0.8                              # warn if SMILES validity collapses below this
RL_OUT_DIR      = "/content/ckpt-mech-rl"

EVAL_LIMIT      = 60  if QUICK_MODE else 196      # oMe-Gold reactions to score
EVAL_MAX_NEW    = 1024                            # generation budget per reaction
OUT_DIR         = "/content/ckpt-mech-sft"        # SFT (DoRA) adapter output
print(dict(QUICK_MODE=QUICK_MODE, BASE_MODEL=BASE_MODEL, DORA=USE_DORA, CURRICULUM=CURRICULUM,
           EPOCHS=EPOCHS, RUN_RL=RUN_RL, RL_ALGO=RL_ALGO, RL_REWARD=RL_REWARD,
           MAX_TRAIN=MAX_TRAIN, EVAL_LIMIT=EVAL_LIMIT))

## 8 · Load base model + tokenizer  ·  ⏱️ 1–3 min (first download)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if BF16 else torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
)
model.config.use_cache = False
print("loaded", BASE_MODEL, "| params:", sum(p.numel() for p in model.parameters())/1e9, "B")

## 9 · Train — DoRA SFT (+ curriculum)  ·  ⏱️ T4: QUICK ~20 min / full ~60–110 min · A100/L4: QUICK ~6 / full ~12–25 min

Robust to TRL version drift: plain `transformers.Trainer` with **prompt-masked labels**
(loss only on the assistant mechanism tokens). Two SOTA levers vs the old cell:
- **DoRA** (`use_dora=True`, `USE_DORA` in cell 7) — weight-decomposed LoRA, a drop-in upgrade;
- **curriculum** (`CURRICULUM`) — rows are ordered easy→medium→hard and shuffling is disabled so
  the model sees simple mechanisms before hard multi-step ones.

Watch the loss fall — a healthy run drops from ~1.5 to ~0.2–0.4.

In [ ]:
import torch
from dataclasses import dataclass
from typing import List, Dict
from torch.utils.data import Dataset, SequentialSampler
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel

# ---- compat shim ----------------------------------------------------------
# transformers 4.55 calls `accelerator.unwrap_model(model, keep_torch_compile=...)`,
# a kwarg only present in accelerate >= 1.3. If an older accelerate is live in the
# runtime (common on Colab when the runtime wasn't restarted after cell 2), wrap
# unwrap_model so the extra kwarg is swallowed. No-op when accelerate is new enough.
import inspect
from accelerate import Accelerator
if "keep_torch_compile" not in inspect.signature(Accelerator.unwrap_model).parameters:
    _orig_unwrap = Accelerator.unwrap_model
    def _unwrap_model(self, model, keep_torch_compile=None, **kw):
        return _orig_unwrap(self, model, **kw)
    Accelerator.unwrap_model = _unwrap_model
    import accelerate as _acc
    print(f"[compat] patched Accelerator.unwrap_model (accelerate {_acc.__version__} "
          "predates keep_torch_compile). For a clean fix, re-run cell 2 and RESTART the runtime.")
# ---------------------------------------------------------------------------

# ---- curriculum: order easy -> medium -> hard (Feature 5) ----
# Reuse the repo helper so notebook + CLI behave identically.
from training.curriculum import order_by_difficulty, level_histogram
_train = train_rows[:MAX_TRAIN] if MAX_TRAIN else list(train_rows)
if CURRICULUM:
    _train = order_by_difficulty(_train)     # stable easy->medium->hard->unknown
    print("curriculum on | order:", level_histogram(_train))

# ---- prompt-masked tokenization (loss on assistant span only) ----
def encode(example):
    msgs = example["messages"]
    prompt_ids = tok.apply_chat_template(msgs[:-1], add_generation_prompt=True, tokenize=True)
    full_ids   = tok.apply_chat_template(msgs,      add_generation_prompt=False, tokenize=True)
    labels = [-100]*len(prompt_ids) + full_ids[len(prompt_ids):]
    full_ids, labels = full_ids[:MAX_SEQ_LEN], labels[:MAX_SEQ_LEN]
    return {"input_ids": full_ids, "labels": labels}

class ChatDS(Dataset):
    def __init__(self, rows): self.rows = [encode(r) for r in rows]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

@dataclass
class Collator:
    pad_id: int
    def __call__(self, feats: List[Dict]):
        m = max(len(f["input_ids"]) for f in feats)
        import torch
        ids, lab, att = [], [], []
        for f in feats:
            n = m - len(f["input_ids"])
            ids.append(f["input_ids"] + [self.pad_id]*n)
            lab.append(f["labels"]    + [-100]*n)
            att.append([1]*len(f["input_ids"]) + [0]*n)
        return {"input_ids": torch.tensor(ids), "labels": torch.tensor(lab),
                "attention_mask": torch.tensor(att)}

train_ds, val_ds = ChatDS(_train), ChatDS(val_rows)
print(f"tokenized train={len(train_ds)} val={len(val_ds)}")

# ---- LoRA / DoRA ----
# Idempotent + portable: skip if an adapter is already attached (safe re-run),
# and name Qwen's projection layers explicitly instead of the "all-linear"
# shorthand (which requires a raw PreTrainedModel and breaks on re-wrap).
if isinstance(model, PeftModel):
    print("A LoRA/DoRA adapter is already attached to `model` — reusing it. "
          "To start fresh, re-run the 'Load base model' cell first.")
else:
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", use_dora=USE_DORA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]))
    print("adapter:", "DoRA" if USE_DORA else "LoRA")
model.print_trainable_parameters()
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

args = TrainingArguments(
    output_dir=OUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, warmup_ratio=0.03, lr_scheduler_type="cosine",
    logging_steps=10, save_strategy="no",
    bf16=BF16, fp16=not BF16, gradient_checkpointing=True,
    report_to="none", optim="adamw_torch",
)

# With curriculum on, force sequential sampling so the easy->hard order is preserved
# (HF Trainer otherwise wraps the train set in a RandomSampler).
class CurriculumTrainer(Trainer):
    def _get_train_sampler(self, *a, **k):
        return SequentialSampler(self.train_dataset)

TrainerCls = CurriculumTrainer if CURRICULUM else Trainer
trainer = TrainerCls(model=model, args=args, train_dataset=train_ds,
                     data_collator=Collator(tok.pad_token_id))
trainer.train()
model.save_pretrained(OUT_DIR); tok.save_pretrained(OUT_DIR)
print("saved SFT adapter ->", OUT_DIR)

## 10 · RL — GRPO / DAPO against the verifiable oMeS reward  ·  ⏱️ T4 QUICK ~25–45 min · A100/L4 QUICK ~8–15 min

**The lever the oMeBench paper left untapped.** We sample `RL_NUM_GEN` mechanisms per reaction,
score each with the repo's **oMeS reward** (`training/reward.py`) — a multiplicative *format gate*
(unparseable/all-invalid → penalty, oMeS skipped) plus an explicit *validity* term and *format*
bonus — and optimize the policy toward higher partial scores. A **validity-collapse monitor**
(`training/callbacks.py`) logs mean reward / S_partial / validity / length each step and warns if
validity dips below `VALIDITY_FLOOR`.

- `RL_ALGO="dapo"` → token-level loss + clip-higher (`epsilon_high=0.28`) + truncated-completion
  masking (better for long CoT); `"grpo"` → stock GRPO. Knobs are filtered to what the installed
  TRL supports (`training/algo.py`), so this degrades gracefully.
- We **merge the SFT (DoRA) adapter into the base weights** so RL starts from the SFT policy, then
  attach a fresh LoRA adapter for the RL updates.

Set `RUN_RL=False` (cell 7) to skip and evaluate the SFT model only.

In [ ]:
rl_model_dir = OUT_DIR   # default: eval the SFT adapter if RL is skipped

if not RUN_RL:
    print("RL skipped (RUN_RL=False). Cell 11 will evaluate the SFT adapter at", OUT_DIR)
else:
    import gc, torch, json
    from datasets import Dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import GRPOConfig, GRPOTrainer
    from peft import LoraConfig, PeftModel

    from training.algo import build_algo_config_kwargs, filter_supported_kwargs
    from training.reward import build_reward_funcs
    from training.callbacks import MechMonitor, instrument_reward_funcs, make_monitor_callback
    from training.curriculum import order_by_difficulty

    # 1) Merge the SFT (DoRA) adapter into base weights so RL starts from the SFT policy.
    #    Free the SFT trainer/model first to reclaim VRAM.
    try:
        del trainer
    except NameError:
        pass
    if isinstance(model, PeftModel):
        print("merging SFT adapter into base weights ...")
        model = model.merge_and_unload()
    merged_dir = "/content/ckpt-mech-sft-merged"
    model.save_pretrained(merged_dir); tok.save_pretrained(merged_dir)
    del model; gc.collect(); torch.cuda.empty_cache()

    # 2) RL dataset: GRPO needs 'prompt' (chat) + reward-func kwarg columns ('reference').
    rl_rows = list(train_rows)
    if CURRICULUM:
        rl_rows = order_by_difficulty(rl_rows)
    if RL_TRAIN_ROWS:
        rl_rows = rl_rows[:RL_TRAIN_ROWS]
    rl_ds = Dataset.from_list([{"prompt": r["prompt"], "reference": r["reference"]} for r in rl_rows])
    print(f"RL prompts: {len(rl_ds)} | algo={RL_ALGO} reward={RL_REWARD} num_gen={RL_NUM_GEN}")

    # 3) Reward funcs (gated oMeS + validity + format) + validity-collapse monitor.
    reward_funcs = build_reward_funcs(reward=RL_REWARD)
    monitor = MechMonitor(validity_floor=VALIDITY_FLOOR)
    reward_funcs = instrument_reward_funcs(reward_funcs, monitor)
    monitor_cb = make_monitor_callback(monitor, raise_on_floor=False)

    # 4) Algorithm knobs (GRPO vs DAPO), filtered to what this TRL supports.
    base_cfg = dict(
        output_dir=RL_OUT_DIR, num_generations=RL_NUM_GEN,
        learning_rate=RL_LR, per_device_train_batch_size=RL_NUM_GEN,
        gradient_accumulation_steps=2, max_steps=RL_STEPS,
        max_prompt_length=RL_MAX_PROMPT, max_completion_length=RL_MAX_COMPL,
        temperature=1.0, beta=0.02, logging_steps=2, save_steps=10_000,
        bf16=BF16, fp16=not BF16, shuffle_dataset=not CURRICULUM,
        report_to="none",
    )
    algo_kwargs = build_algo_config_kwargs(algo=RL_ALGO, epsilon_high=0.28,
                                           dynamic_sampling=True, mask_truncated=True)
    cfg = GRPOConfig(**filter_supported_kwargs({**base_cfg, **algo_kwargs}, GRPOConfig))

    # 5) Fresh LoRA adapter for the RL updates, on the merged SFT policy.
    peft_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"])

    rl_trainer = GRPOTrainer(
        model=merged_dir, args=cfg, train_dataset=rl_ds,
        processing_class=tok, reward_funcs=reward_funcs,
        peft_config=peft_config, callbacks=[monitor_cb])
    rl_trainer.train()
    rl_trainer.save_model(RL_OUT_DIR); tok.save_pretrained(RL_OUT_DIR)
    rl_model_dir = RL_OUT_DIR

    # 6) Reload the RL policy (base + RL adapter) for evaluation in cell 11.
    del rl_trainer; gc.collect(); torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained(
        merged_dir, torch_dtype=torch.bfloat16 if BF16 else torch.float16,
        device_map={"": 0}, trust_remote_code=True)
    model = PeftModel.from_pretrained(model, RL_OUT_DIR)
    model.eval()
    print("RL done. Monitor snapshot:", json.dumps(monitor.snapshot()))
    print("RL policy (for cell 11) ->", rl_model_dir)

## 11 · Evaluate the fine-tuned model on oMe-Gold  ·  ⏱️ T4 QUICK ~8 min / full ~25–40 min

Scores whatever is in `model` — the **RL policy** if cell 10 ran, else the **SFT** model — on the
held-out gold set with the repo's **exact oMeS scorer** (`omebench_eval.scoring.oMeS`) and the same
`default.txt` prompt the benchmark uses. Batched greedy generation; the metric is identical to the
paper's, so `S_partial` here is comparable to the frontier table below.

In [ ]:
import json, torch, time
from omebench_eval.scoring import oMeS, canonical_smiles
from omebench_eval.parsing import extract_mechanism
from omebench_eval.dataset import load_dataset, load_prompt_template, build_prompt

SYSTEM = "You are an expert in organic reaction mechanisms."
gold = load_dataset("gold")[:EVAL_LIMIT]
template = load_prompt_template("default")

def score_model(m, limit_rows, batch=8, max_new=EVAL_MAX_NEW, label="model"):
    m.eval(); tok.padding_side = "left"
    prompts, ids = [], []
    for r in limit_rows:
        u = build_prompt(template, r["reactants_smiles"], r["products_smiles"], r.get("conditions"))
        text = tok.apply_chat_template(
            [{"role":"system","content":SYSTEM},{"role":"user","content":u}],
            tokenize=False, add_generation_prompt=True)
        prompts.append(text); ids.append(r["reaction_id"])
    outs = {}
    t0 = time.time()
    for i in range(0, len(prompts), batch):
        chunk = prompts[i:i+batch]
        enc = tok(chunk, return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_SEQ_LEN).to(m.device)
        with torch.no_grad():
            gen = m.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
        for j, seq in enumerate(gen):
            new = seq[enc.input_ids.shape[1]:]
            outs[ids[i+j]] = tok.decode(new, skip_special_tokens=True)
        print(f"  [{label}] {min(i+batch,len(prompts))}/{len(prompts)}  "
              f"({time.time()-t0:.0f}s)", end="\r")
    print()
    # score with oMeS
    per = []
    for r in limit_rows:
        mech = extract_mechanism(outs.get(r["reaction_id"]))
        if not isinstance(mech, list):
            per.append({"level": r.get("level"), "S_total":0.0,"S_partial":0.0,"V":0,"L":0}); continue
        pred = [(s.get("subtype"), s.get("intermediate_smiles","")) for s in mech if isinstance(s,dict)]
        g = [(s["subtype"], s["intermediate_smiles"], s["step_weight"]) for s in r["mechanism"]]
        try: res = oMeS(g, pred); per.append({"level":r.get("level"),"S_total":res.S_total,
                                              "S_partial":res.S_partial,"V":res.V,"L":res.L})
        except Exception: per.append({"level":r.get("level"),"S_total":0.0,"S_partial":0.0,"V":0,"L":0})
    def avg(k, rows=per): return round(sum(x[k] for x in rows)/max(1,len(rows)),4)
    summary = {"model":label,"n":len(per),"S_partial":avg("S_partial"),"S_total":avg("S_total"),
               "V":avg("V"),"L":avg("L")}
    for lvl in ("easy","medium","hard"):
        sub=[x for x in per if x["level"]==lvl]
        if sub: summary[f"S_partial_{lvl}"]=round(sum(x['S_partial'] for x in sub)/len(sub),4)
    return summary

_stage = "sft+rl" if RUN_RL else "sft"
ft = score_model(model, gold, label=f"qwen-mech-{_stage}")
print("\nFINE-TUNED:", json.dumps(ft, indent=2))

## 12 · (optional) Evaluate the **base** model for a delta  ·  ⏱️ same as #11
Shows how much SFT+RL moved the needle. Reloads the base weights (no adapter).
Skip to save time if you only need the leaderboard slot.

In [ ]:
RUN_BASE_EVAL = True   # set False to skip

base_summary = None
if RUN_BASE_EVAL:
    from transformers import AutoModelForCausalLM
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16 if BF16 else torch.float16,
        device_map={"":0}, trust_remote_code=True)
    base_summary = score_model(base, gold, label=f"{BASE_MODEL.split('/')[-1]} (base)")
    print("\nBASE:", json.dumps(base_summary, indent=2))
    del base; torch.cuda.empty_cache()

## 13 · Leaderboard vs frontier models  ·  ⏱️ <5 s

Frontier oMeS **S_partial on oMe-Gold** (0–1 scale). Sources: oMeBench paper (arXiv:2510.07731)
and this repo's own API-harness runs (see `README.md` / `BRAINLIFT.md`). Our fine-tuned row (SFT
or SFT+RL, per cell 7) is inserted from cell 11.

> Reality check (from the paper's feasibility notes): a **1.5 B** specialist won't beat GPT-5.x;
> the honest target is to **clear the weak/mid baselines** (GPT-4o 0.05, Sonnet-4 0.18) and approach
> the paper's 4 B specialist (~0.20–0.30) on in-domain mechanisms — while *proving no test leakage*.
> RL (cell 10) is the lever expected to move S_partial above the SFT-only number.

In [ ]:
FRONTIER = [
    ("Gemini-3.1-Pro",        0.51,  "paper/README"),
    ("GPT-5.5 (this harness)",0.469, "repo run"),
    ("Gemini-Pro-2.5",        0.379, "paper"),
    ("GPT-5",                 0.291, "paper"),
    ("o3",                    0.281, "paper"),
    ("paper 4B SFT specialist",0.30, "paper (ICL)"),
    ("DeepSeek-R1 (685B)",    0.25,  "paper"),
    ("Claude-Sonnet-4",       0.179, "paper"),
    ("GPT-4o",                0.05,  "paper"),
    ("Qwen-3-4B (untuned)",   0.042, "paper"),
    ("LLaMA-3-8B (untuned)",  0.033, "paper"),
]
board = list(FRONTIER)
board.append((ft["model"] + "  ⭐ (ours)", ft["S_partial"], f"this notebook, n={ft['n']}"))
if base_summary:
    board.append((base_summary["model"], base_summary["S_partial"], f"this notebook, n={base_summary['n']}"))
board.sort(key=lambda x: -x[1])

print(f"{'model':<34}{'S_partial':>10}   source")
print("-"*70)
for name, sp, src in board:
    star = "  <<<" if "ours" in name else ""
    print(f"{name:<34}{sp:>10.3f}   {src}{star}")

print("\nOur model, by difficulty:")
for lvl in ("easy","medium","hard"):
    k=f"S_partial_{lvl}"
    if k in ft: print(f"  {lvl:<7} {ft[k]:.3f}")
print(f"\nValidity(V)={ft['V']}  LogicalFidelity(L)={ft['L']}  S_total={ft['S_total']}")

## 14 · (optional) Live frontier eval via API  ·  ⏱️ ~5–15 min
Reproduce a frontier number yourself instead of trusting the table. Needs an API key and spend.
Uses the repo's API harness (identical oMeS scorer).

In [ ]:
RUN_API_EVAL = False   # set True and add a key

if RUN_API_EVAL:
    import os, subprocess
    os.environ["OPENAI_API_KEY"] = ""     # <-- your key
    # or: os.environ["ANTHROPIC_API_KEY"] = "..."
    %pip -q install openai anthropic
    # score 40 gold reactions with gpt-5.5 (edit --models / --limit as you like)
    !python -m omebench_eval.cli run --models gpt-5.5 --dataset gold --prompt cot --limit 40 --max-tokens 16000
    !python -m omebench_eval.cli report --dataset gold

## Next steps & honesty notes

This notebook now runs the **full SFT → RL** recipe end to end. To push further:

- **Turn on distillation** (cell 6, `RUN_DISTILL=True`) — warm-starting SFT on frontier CoT that
  *passes the oMeS verifier* is the single biggest lever (ether0, RetroDFM-R).
- **Full run**: set `QUICK_MODE=False` (more data + epochs), raise `RL_STEPS`/`RL_NUM_GEN`, and
  bump `MAX_SEQ_LEN` toward the CLI default of 6000 if cell 5's p99 is high.
- **Round-trip reward**: set `RL_REWARD="omes+roundtrip"` and wire a real forward model
  (`training/forward_model.py`) — it's a no-op until you supply one, so it never hurts.
- **Scale/verify at the CLI**: `training/sft_train.py` and `training/grpo_train.py` expose the same
  flags (`--dora`, `--curriculum`, `--algo`, `--reward`, `--validity-floor`) for multi-GPU / vLLM
  runs; see `docs/sota_features.md`.

**Honesty notes**
- We evaluate on **oMe-Gold**, held out and *proven* decontaminated (cells 4/5). No leakage — and
  every augmented/distilled row is decontaminated too.
- RL is expected to *lift* S_partial over SFT, but on a 1.5 B model with a QUICK run the gain may be
  small or noisy; the `VALIDITY_FLOOR` monitor guards against the RL validity-collapse failure mode.
- oMeBench measures *mechanism* reasoning specifically. Broad chemistry knowledge (ChemBench, GPQA)
  is out of scope for a 1.5 B mechanism specialist.
- In-domain scores (same named reactions, new substituents) run higher than truly novel mechanisms —
  interpret a leaderboard win as *data/param-efficiency*, not general chemical mastery.
